# 02a – Player Team Dashboard: Relación de métricas con Victoria/Impacto

Este cuaderno revisa el dataset de *player dashboard by team* para detectar cómo las métricas individuales se relacionan con las victorias y el impacto en cancha.

**Columnas clave**: `W_PCT`, `PLUS_MINUS`, `W`, `L`, `PTS`, `MIN`, `FG_PCT`, `AST`, `REB`, `TOV`.

**Ruta analítica**
1. Carga y validación de datos.
2. Preparación del dataset.
3. Correlaciones principales.
4. Visualizaciones concisas.
5. Exportación de resultados y conclusión.


## 1. Configuración inicial
Breve preparación del entorno de trabajo con librerías estándar y rutas editables.


In [ ]:
from pathlib import Path
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path("../../../00_data/00c_final/2024-25/dashboards/team_player_dashboard__dataset_1.csv")
BASE_OUTPUT_DIR = Path("02a_dasboard_outs/02a_dashboard")
FIGURE_DIR = BASE_OUTPUT_DIR / "figures"
TABLE_DIR = BASE_OUTPUT_DIR / "tables"

SAVE_FIG = True
SAVE_TABLE = True

SAVED_FIGS = []
SAVED_TABLES = []

np.random.seed(42)


## 2. Carga y validación de datos
Se cargan los datos desde CSV (o Parquet) y se revisa que existan las columnas imprescindibles (`TEAM_NAME`, `PLAYER_NAME`, `W_PCT`).


In [ ]:
if not CSV_PATH.exists():
    warnings.warn(f"No se encontró el archivo en {CSV_PATH}.")
    raise SystemExit("Sin archivo de entrada no se puede continuar.")

if CSV_PATH.suffix.lower() == ".parquet":
    df = pd.read_parquet(CSV_PATH)
else:
    df = pd.read_csv(CSV_PATH)

required_cols = {"TEAM_NAME", "PLAYER_NAME", "W_PCT"}
missing = required_cols.difference(df.columns)

print("Dimensiones iniciales:", df.shape)
print("Columnas disponibles:", len(df.columns))
print(df.head(3))
print(df.dtypes.head(10))

if missing:
    warnings.warn(f"Faltan columnas imprescindibles: {sorted(missing)}")
    raise SystemExit("Análisis detenido por columnas faltantes.")

## 3. Preparación del dataset
Se tipifican columnas, se armonizan identificadores duplicados (`TEAM_ID`, `team_id`) y se convierten las métricas numéricas para optimizar el análisis.


In [ ]:
df.columns = [c.strip() for c in df.columns]

if "team_id" in df.columns and "TEAM_ID" in df.columns:
    df["TEAM_ID"] = df["TEAM_ID"].fillna(df["team_id"])
if "team_id" in df.columns and "TEAM_ID" not in df.columns:
    df.rename(columns={"team_id": "TEAM_ID"}, inplace=True)

if "season" in df.columns and "SEASON_YEAR" in df.columns:
    df["SEASON_YEAR"] = df["SEASON_YEAR"].fillna(df["season"])
if "season" not in df.columns and "SEASON_YEAR" in df.columns:
    df.rename(columns={"SEASON_YEAR": "season"}, inplace=True)

if "dataset" in df.columns:
    df = df[df["dataset"] == 1].copy()

cat_candidates = [
    "TEAM_ID", "TEAM_NAME", "PLAYER_ID", "PLAYER_NAME", "NICKNAME",
    "GROUP_SET", "season", "SEASON_YEAR", "season_type", "endpoint"
]
cat_cols = [c for c in cat_candidates if c in df.columns]
rank_cols = [c for c in df.columns if c.endswith("_RANK")]
target_cols = [c for c in ["W", "L", "W_PCT", "PLUS_MINUS"] if c in df.columns]
num_cols = [c for c in df.columns if c not in cat_cols]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

import io
buffer = io.StringIO()
df.info(buf=buffer, verbose=False)
info_lines = buffer.getvalue().splitlines()
print("Resumen de tipos (primeras líneas):")
print("
".join(info_lines[:12]))

print("Columnas categóricas:", cat_cols)
print("Columnas objetivo:", target_cols)
print("Columnas numéricas:", len(num_cols))

## 4. Correlaciones principales
Se calcula la correlación de Spearman entre cada métrica numérica y los indicadores de rendimiento (`W_PCT`, `PLUS_MINUS`, `W`, `L`). Se listan solo los indicadores con mayor magnitud (hasta 12 por objetivo).


In [ ]:
corr_summary = {}
max_features = 12

for target in target_cols:
    candidates = [c for c in num_cols if c != target]
    if not candidates:
        continue
    series = df[candidates].corrwith(df[target], method="spearman")
    corr_df = series.dropna().to_frame(name="spearman").sort_values(
        by="spearman", key=lambda s: s.abs(), ascending=False
    )
    corr_summary[target] = corr_df
    print(f"Correlaciones destacadas con {target}:")
    print(corr_df.head(max_features))
    print()

## 5. Visualizaciones simples y limpias
Se emplean pocas figuras con estética uniforme (tipografía tipo LaTeX) y se guardan en `02a_dasboard_outs/02a_dashboard/figures/`.


In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})

def save_figure(fig, filename):
    filename = filename.replace(" ", "_").lower()
    path = FIGURE_DIR / filename
    if SAVE_FIG:
        fig.savefig(path, dpi=120, bbox_inches="tight")
        SAVED_FIGS.append(path.name)
    plt.close(fig)

### 5.1 Heatmap de correlaciones relevantes
Comparativa visual entre las métricas con mayor magnitud de correlación y los objetivos de victoria/impacto.


In [ ]:
heat_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in corr_summary]
heat_features = []
for target in heat_targets:
    if target in corr_summary:
        heat_features.extend(corr_summary[target].head(10).index.tolist())
heat_features = sorted(set(heat_features))

if heat_targets and heat_features:
    import numpy as np
    matrix = []
    for feature in heat_features:
        row = []
        for target in heat_targets:
            value = corr_summary[target].loc[feature, "spearman"] if feature in corr_summary[target].index else np.nan
            row.append(value)
        matrix.append(row)
    matrix = np.array(matrix)
    fig, ax = plt.subplots()
    im = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(heat_targets)))
    ax.set_xticklabels(heat_targets)
    ax.set_yticks(range(len(heat_features)))
    ax.set_yticklabels(heat_features)
    ax.set_title("Correlaciones de Spearman destacadas")
    cbar = fig.colorbar(im, ax=ax, shrink=0.85)
    cbar.ax.set_ylabel("Spearman", rotation=270, labelpad=15)
    save_figure(fig, "heatmap_correlaciones.png")
else:
    print("No hay suficientes métricas para construir el heatmap.")

### 5.2 Dispersión del indicador más asociado a W_PCT
Revisión puntual del predictor con mayor magnitud de correlación frente al porcentaje de victorias.


In [ ]:
if "W_PCT" in corr_summary and not corr_summary["W_PCT"].empty:
    top_feature = corr_summary["W_PCT"].index[0]
    plot_df = df[[top_feature, "W_PCT"]].dropna()
    if not plot_df.empty:
        fig, ax = plt.subplots()
        if len(plot_df) > 800:
            hb = ax.hexbin(plot_df[top_feature], plot_df["W_PCT"], gridsize=30, cmap="viridis", mincnt=1)
            fig.colorbar(hb, ax=ax, shrink=0.85, label="Frecuencia")
        else:
            ax.scatter(plot_df[top_feature], plot_df["W_PCT"], s=18, alpha=0.65, color="#1f77b4", edgecolor="none")
        ax.set_xlabel(top_feature)
        ax.set_ylabel("W_PCT")
        ax.set_title(f"{top_feature} vs W_PCT")
        ax.grid(alpha=0.2)
        save_figure(fig, f"scatter_{top_feature}_vs_w_pct.png")
    else:
        print("Sin datos válidos para el scatter.")
else:
    print("No se encontraron correlaciones para W_PCT.")

### 5.3 Barras: jugadores extremos por PLUS_MINUS
Los jugadores con mejor y peor `PLUS_MINUS` se representan para detectar outliers rápidamente.


In [ ]:
if {"PLAYER_NAME", "PLUS_MINUS"}.issubset(df.columns):
    cols = [c for c in ["PLAYER_NAME", "TEAM_NAME", "PLUS_MINUS", "MIN"] if c in df.columns]
    players_df = df[cols].dropna(subset=["PLUS_MINUS"]).copy()
    if not players_df.empty:
        top_players = players_df.sort_values("PLUS_MINUS", ascending=False).head(8)
        bottom_players = players_df.sort_values("PLUS_MINUS", ascending=True).head(8)
        summary = pd.concat([top_players, bottom_players])
        labels = summary["PLAYER_NAME"].astype(str)
        if "TEAM_NAME" in summary.columns:
            labels = labels + " (" + summary["TEAM_NAME"].fillna("-") + ")"
        positions = np.arange(len(summary))
        colors = ["#1b7837" if val >= 0 else "#b2182b" for val in summary["PLUS_MINUS"]]
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(positions, summary["PLUS_MINUS"], color=colors)
        ax.set_yticks(positions)
        ax.set_yticklabels(labels)
        ax.axvline(0, color="#444444", linewidth=1)
        ax.set_xlabel("PLUS_MINUS")
        ax.set_title("Jugadores con PLUS_MINUS extremos")
        fig.tight_layout()
        save_figure(fig, "barras_plus_minus_extremos.png")
    else:
        print("Sin información suficiente para el gráfico de barras.")
else:
    print("No están disponibles las columnas para el gráfico de jugadores.")

## 6. Exportación de resultados
Se guardan tablas de correlaciones y se reporta cuántos artefactos se generaron en `figures/` y `tables/`.


In [ ]:
if SAVE_TABLE:
    for target, corr_df in corr_summary.items():
        table_path = TABLE_DIR / f"correlaciones_{target.lower()}.csv"
        corr_df.to_csv(table_path)
        SAVED_TABLES.append(table_path.name)

print(f"Figuras guardadas: {len(SAVED_FIGS)}")
print(f"Tablas guardadas: {len(SAVED_TABLES)}")

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M")
VERSIONS = {
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": plt.__version__ if hasattr(plt, "__version__") else "N/D"
}
print("Versiones registradas:", VERSIONS)
print("Fecha de ejecución:", RUN_TIMESTAMP)

## 7. Conclusión
Las correlaciones muestran qué métricas individuales acompañan con mayor consistencia a las victorias: 
- Priorizar los indicadores con mayor magnitud en la tabla de correlaciones de Spearman (habitualmente porcentajes de tiro, asistencias y diferencial de plus/minus).
- Los gráficos revelan rápidamente los outliers: los jugadores destacados por `PLUS_MINUS` extremo merecen análisis cualitativo adicional.

**Diferencias clave:** las métricas ofensivas eficientes y la disciplina (bajos `TOV`) suelen alinearse con mayor `W_PCT`, mientras que los registros negativos concentrados en ciertos jugadores apuntan a ajustes de rotación.

**Próximos pasos:** extender este flujo a otros datasets (`dataset` 2, análisis por posición) y combinarlo con información contextual de rivalidades o calendarios.

**Versiones y trazabilidad**
- pandas: `pd.__version__`
- numpy: `np.__version__`
- matplotlib: `plt.__version__`
- Fecha y hora de ejecución: consulte la impresión en la celda anterior (`RUN_TIMESTAMP`).
